# LSTM Baseline Validation — ESC-50 Environmental Sound Classification

**Purpose.** Smoke-test the full LSTM pipeline on **both segment variants** before launching the experiment grid. This notebook runs the **Run 1 baseline architecture** (hidden=128, 2 layers, bidirectional, final pooling) on fold 1 only for:

* `short` — 41 frames, 50% overlap (paper SM / SP regime)
* `long` — 101 frames, 90% overlap (paper LM / LP regime)

If both variants train without NaN and clip-level accuracy exceeds Piczak's 0.44 baseline "B", the pipeline is healthy and we can proceed to the full 9-architecture grid, 5-fold promotion, and optional long-variant stages in **`LSTM experiments.ipynb`**.

**Run-name convention.** The segment variant is **always** suffixed onto the run name (e.g. `LSTM_Run1_Baseline_short`, `LSTM_Run1_Baseline_long`) so artifacts never collide between short and long.

Shared data pipeline (Steps 1–7) is identical to the Piczak baseline notebook. LSTM-specific code begins at **Step 8**.

See `LSTM Experiments.md` for the full design rationale, expected ranges, and long-variant notes.

In [6]:
# Step 1: Imports and configuration

%pip install -q librosa

import os
import time
import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from google.colab import drive
drive.mount('/content/drive')

SEED = 99
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ZIP_PATH    = '/content/drive/MyDrive/ESC Project Deep Learning 5888 Data Dump/piczak_dataset.zip'
EXTRACT_DIR = '/content/piczak_dataset'
OUTPUT_DIR  = '/content/drive/MyDrive/ESC Project Deep Learning 5888 Data Dump/LSTM'

os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.isdir(EXTRACT_DIR):
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_file:
        zip_file.extractall(EXTRACT_DIR)

CSV_PATH  = None
AUDIO_DIR = None

for root, dirs, files in os.walk(EXTRACT_DIR):
    if 'esc50.csv' in files:
        CSV_PATH = os.path.join(root, 'esc50.csv')
    wav_files = [f for f in files if f.endswith('.wav')]
    if wav_files and AUDIO_DIR is None:
        AUDIO_DIR = root

print('Dataset CSV:  ', CSV_PATH)
print('Audio folder: ', AUDIO_DIR)
print('Output folder:', OUTPUT_DIR)
print('Device:       ', DEVICE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset CSV:   /content/piczak_dataset/esc50.csv
Audio folder:  /content/piczak_dataset/audio/audio
Output folder: /content/drive/MyDrive/ESC Project Deep Learning 5888 Data Dump/LSTM
Device:        cuda


In [8]:
# Step 2: Shared spectrogram hyperparameters
# These match the Piczak baseline exactly — do not change.

SAMPLE_RATE = 22050
N_FFT       = 1024
HOP_LENGTH  = 512
N_MELS      = 60
NUM_CLASSES = 50

VARIANT_CONFIGS = {
    'short': {
        'segment_frames':     41,
        'segment_hop_frames': 20,
    },
    'long': {
        'segment_frames':     101,
        'segment_hop_frames': 10,
    },
}

In [9]:
# Step 3: Load ESC-50 metadata

df = pd.read_csv(CSV_PATH)
display(df.head())

,filename,fold,target,category,esc10,src_file,take
0,1-100032-A-0.wav,1,0,dog,True,100032,A
1,1-100038-A-14.wav,1,14,chirping_birds,False,100038,A
2,1-100210-A-36.wav,1,36,vacuum_cleaner,False,100210,A
3,1-100210-B-36.wav,1,36,vacuum_cleaner,False,100210,B
4,1-101296-A-19.wav,1,19,thunderstorm,False,101296,A


In [10]:
# Step 4: Feature extraction — log-mel spectrogram + delta

def extract_features(audio_path, sr=SAMPLE_RATE, n_fft=N_FFT,
                     hop_length=HOP_LENGTH, n_mels=N_MELS):
    """Load one ESC-50 clip and return a (2, N_MELS, T) array:
       channel 0 = log-mel spectrogram
       channel 1 = delta
    """
    y, _ = librosa.load(audio_path, sr=sr)
    y    = librosa.util.normalize(y)

    mel_spec = librosa.feature.melspectrogram(
        y=y, sr=sr, n_fft=n_fft, hop_length=hop_length,
        n_mels=n_mels, power=2.0
    )
    log_mel = librosa.power_to_db(mel_spec)
    delta   = librosa.feature.delta(log_mel)

    return np.stack([log_mel, delta], axis=0).astype(np.float32)

In [11]:
# Step 5: Segment extraction

def extract_segments(features, segment_frames, segment_hop_frames):
    """Extract overlapping spectrogram segments; discard silent ones."""
    n_frames = features.shape[2]
    segments = []
    for start in range(0, n_frames - segment_frames + 1, segment_hop_frames):
        segment = features[:, :, start:start + segment_frames]
        if np.mean(np.abs(segment[0])) > 1e-6:
            segments.append(segment.astype(np.float32))
    return segments

In [12]:
# Step 6: Build segment samples and Dataset

def build_segment_samples(dataframe, audio_dir, segment_frames, segment_hop_frames):
    samples = []
    for idx in range(len(dataframe)):
        row        = dataframe.iloc[idx]
        audio_path = os.path.join(audio_dir, row['filename'])
        label      = int(row['target'])
        filename   = row['filename']

        features = extract_features(audio_path)
        segments = extract_segments(features, segment_frames, segment_hop_frames)
        for segment in segments:
            samples.append((segment, label, filename))

        if (idx + 1) % 200 == 0:
            print(f'  Processed {idx + 1}/{len(dataframe)} clips...')

    print(f'  Total segments: {len(samples)} from {len(dataframe)} clips')
    return samples


class ESC50Dataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        segment, label, filename = self.samples[idx]
        return torch.from_numpy(segment).float(), label, filename

In [13]:
# Step 7: Train/test dataloader factory

def get_train_test_loaders(dataframe, audio_dir, test_fold,
                           variant_name='short', batch_size=64, num_workers=2):
    """Returns (train_loader, test_loader) for one fold."""
    train_df = dataframe[dataframe['fold'] != test_fold].reset_index(drop=True)
    test_df  = dataframe[dataframe['fold'] == test_fold].reset_index(drop=True)

    seg_frames = VARIANT_CONFIGS[variant_name]['segment_frames']
    seg_hop    = VARIANT_CONFIGS[variant_name]['segment_hop_frames']

    train_samples = build_segment_samples(train_df, audio_dir, seg_frames, seg_hop)
    test_samples  = build_segment_samples(test_df,  audio_dir, seg_frames, seg_hop)

    train_loader = DataLoader(ESC50Dataset(train_samples), batch_size=batch_size,
                              shuffle=True,  num_workers=num_workers, pin_memory=True)
    test_loader  = DataLoader(ESC50Dataset(test_samples),  batch_size=batch_size,
                              shuffle=False, num_workers=num_workers, pin_memory=True)
    return train_loader, test_loader

---
## LSTM Model

All LSTM-specific code is below. The only thing that changes between experimental runs is `LSTM_CONFIG`.

In [14]:
# Step 8: LSTM experiment config
#
# This is the single dict to edit between runs.
# Architectural factors varied across the 9-run experiment plan:
#   hidden_size  : 64 | 128 | 256
#   num_layers   : 1  | 2   | 3
#   bidirectional: True | False
#   pooling      : 'final' | 'mean'
#
# dropout and learning_rate are held fixed across all runs —
# they are training hyperparameters, not architectural ones.
# Varying them would confound architectural comparisons.
# Exception: if a run produces NaN loss, halve lr to 5e-4
# as a debugging step before concluding the architecture is the problem.

LSTM_CONFIG = {
    # --- identity ---
    'model_name':    'LSTM_Run1_Baseline',
    'variant_name':  'short',
    'save_dir':      OUTPUT_DIR, # Updated to use the variable defined in Step 1

    # --- architectural factors (vary these between runs) ---
    'hidden_size':   128,
    'num_layers':    2,
    'bidirectional': True,
    'pooling':       'final',   # 'final' or 'mean'

    # --- training hyperparameters (fixed across all runs) ---
    'dropout':       0.3,
    'learning_rate': 1e-3,
    'num_epochs':    60,
    'batch_size':    64,
}

In [15]:
# Step 9: LSTM model — config-driven
#
# Input  : (B, 2, 60, T)  — batch, channels, mel_bins, time_frames
# Reshape: (B, T, 120)    — treat each time frame as one sequence step
# Output : (B, num_classes)
#
# Pooling options
#   'final' — concatenate last-layer forward + backward hidden states.
#             Relies on the LSTM to carry relevant info to the last step.
#   'mean'  — average all timestep outputs.
#             More robust when the key sound event occurs mid-clip.

class LSTMBaseline(nn.Module):
    def __init__(self, config):
        super().__init__()
        input_size         = 2 * N_MELS                  # 2 channels * 60 mel bins = 120
        hidden_size        = config['hidden_size']
        num_layers         = config['num_layers']
        dropout            = config['dropout']
        bidirectional      = config['bidirectional']
        self.pooling       = config['pooling']
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size * self.num_directions, NUM_CLASSES)
        self._init_weights()

    def _init_weights(self):
        for name, param in self.lstm.named_parameters():
            if 'weight_ih' in name:
                nn.init.xavier_uniform_(param.data)
            elif 'weight_hh' in name:
                nn.init.orthogonal_(param.data)
            elif 'bias' in name:
                param.data.fill_(0)
                # Forget gate bias = 1.0: encourages remembering across
                # long spans early in training (standard LSTM trick)
                n = param.size(0)
                param.data[n // 4 : n // 2].fill_(1.0)
        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)

    def forward(self, x):
        B, C, M, T = x.shape
        x = x.view(B, C * M, T).permute(0, 2, 1)          # (B, T, 120)
        output, (h_n, _) = self.lstm(x)

        if self.pooling == 'mean':
            h = output.mean(dim=1)                          # (B, hidden * directions)
        else:  # 'final'
            if self.bidirectional:
                h = torch.cat([h_n[-2], h_n[-1]], dim=1)   # (B, hidden * 2)
            else:
                h = h_n[-1]                                 # (B, hidden)

        return self.classifier(self.dropout(h))


def build_lstm_from_config(config, device=DEVICE):
    """Instantiate and move model to device directly from config dict."""
    return LSTMBaseline(config).to(device)


# Quick sanity check
model_test = build_lstm_from_config(LSTM_CONFIG)
dummy      = torch.randn(2, 2, N_MELS, VARIANT_CONFIGS['short']['segment_frames']).to(DEVICE)
print('Input shape: ', dummy.shape)
print('Output shape:', model_test(dummy).shape)
total_params = sum(p.numel() for p in model_test.parameters())
print(f'Parameters:  {total_params:,}')
del model_test, dummy

Input shape:  torch.Size([2, 2, 60, 41])
Output shape: torch.Size([2, 50])
Parameters:  664,114


In [16]:
# Step 10: Training loop
#
# Key implementation notes:
#   - Gradient clipping (max_norm=1.0) guards against exploding gradients,
#     which caused the NaN losses in the Piczak 'long' variant.
#   - Cosine LR decay reduces the learning rate smoothly over training.
#   - Two test metrics tracked per epoch:
#       test_acc       — segment-level accuracy (every segment graded vs.
#                        its clip label independently). Matches the raw
#                        numbers the project notebooks have been reporting.
#       clip_test_acc  — clip-level accuracy via probability voting:
#                        average softmax over all segments of a clip,
#                        then argmax. Matches Piczak's SP/LP evaluation
#                        (paper §3.2, "taking into account the probabilities
#                        predicted for each segment"), which is the number
#                        that's directly comparable to Piczak's 44% B
#                        baseline and ~60-65% CNN results on ESC-50.
#   - best_* / best_epoch track peak generalization rather than the
#     final-epoch value.

def _aggregate_clip_probs(probs, labels_cpu, fnames, store):
    """Accumulate per-clip softmax sums for probability voting."""
    for fn, prob, lab in zip(fnames, probs, labels_cpu):
        if fn not in store['sum']:
            store['sum'][fn] = np.zeros(prob.shape[0], dtype=np.float64)
            store['label'][fn] = int(lab)
        store['sum'][fn] += prob


def _clip_accuracy(store):
    """Resolve accumulated clip probabilities to clip-level accuracy."""
    if not store['sum']:
        return 0.0
    correct = 0
    for fn, prob_sum in store['sum'].items():
        if int(np.argmax(prob_sum)) == store['label'][fn]:
            correct += 1
    return correct / len(store['sum'])


def run_model_lstm(model, train_loader, test_loader, config, device=DEVICE):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(
        model.parameters(),
        lr=config['learning_rate'],
        weight_decay=1e-4,
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=config['num_epochs']
    )

    history = {
        'train_loss':     [],
        'train_acc':      [],
        'test_acc':       [],   # segment-level
        'clip_test_acc':  [],   # clip-level (probability voting)
    }
    best_test_acc      = 0.0
    best_epoch         = 0
    best_clip_test_acc = 0.0
    best_clip_epoch    = 0
    start_time         = time.time()

    for epoch in range(config['num_epochs']):
        # --- train ---
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for xb, yb, _ in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss   = criterion(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            running_loss += loss.item() * xb.size(0)
            correct      += (logits.argmax(1) == yb).sum().item()
            total        += yb.size(0)

        scheduler.step()
        train_loss = running_loss / total
        train_acc  = correct / total

        # --- evaluate (segment-level AND clip-level via probability voting) ---
        model.eval()
        seg_correct, seg_total = 0, 0
        clip_store = {'sum': {}, 'label': {}}

        with torch.no_grad():
            for xb, yb, fnames in test_loader:
                xb, yb = xb.to(device), yb.to(device)
                logits = model(xb)
                probs  = torch.softmax(logits, dim=1).cpu().numpy()

                seg_correct += (logits.argmax(1) == yb).sum().item()
                seg_total   += yb.size(0)

                _aggregate_clip_probs(probs, yb.cpu().numpy(), fnames, clip_store)

        test_acc      = seg_correct / seg_total
        clip_test_acc = _clip_accuracy(clip_store)

        # --- track best (separate peaks for segment- and clip-level) ---
        if test_acc > best_test_acc:
            best_test_acc = test_acc
            best_epoch    = epoch + 1
        if clip_test_acc > best_clip_test_acc:
            best_clip_test_acc = clip_test_acc
            best_clip_epoch    = epoch + 1

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)
        history['clip_test_acc'].append(clip_test_acc)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(
                f'[{config["model_name"]}] Epoch {epoch+1:03d}/{config["num_epochs"]} | '
                f'loss {train_loss:.4f} | train {train_acc:.4f} | '
                f'seg {test_acc:.4f} | clip {clip_test_acc:.4f} | '
                f'best clip {best_clip_test_acc:.4f} (ep {best_clip_epoch})'
            )

    results = {
        'model':                 config['model_name'],
        'hidden_size':           config['hidden_size'],
        'num_layers':            config['num_layers'],
        'bidirectional':         config['bidirectional'],
        'pooling':               config['pooling'],
        'final_train_loss':      history['train_loss'][-1],
        'final_train_acc':       history['train_acc'][-1],
        'final_test_acc':        history['test_acc'][-1],        # segment-level
        'final_clip_test_acc':   history['clip_test_acc'][-1],   # clip-level
        'best_test_acc':         best_test_acc,                  # segment-level peak
        'best_epoch':            best_epoch,
        'best_clip_test_acc':    best_clip_test_acc,             # clip-level peak (paper-comparable)
        'best_clip_epoch':       best_clip_epoch,
        'training_time_secs':    round(time.time() - start_time, 1),
    }
    return history, results

In [17]:
# Step 11: Results saving (matches shared team convention: save_dir/model_name)

def save_results(history, results, config):
    save_path = Path(config['save_dir']) / config['model_name']
    save_path.mkdir(parents=True, exist_ok=True)

    pd.DataFrame(history).to_csv(save_path / 'history.csv', index=False)
    pd.DataFrame([results]).to_csv(save_path / 'results.csv', index=False)

    with open(save_path / 'config.json', 'w') as f:
        json.dump(config, f, indent=2)

    plt.figure(figsize=(10, 4))
    epochs = range(1, len(history['train_loss']) + 1)
    plt.plot(epochs, history['train_loss'],    label='Train Loss')
    plt.plot(epochs, history['train_acc'],     label='Train Acc')
    plt.plot(epochs, history['test_acc'],      label='Test Acc (segment)')
    if 'clip_test_acc' in history:
        plt.plot(epochs, history['clip_test_acc'], label='Test Acc (clip, prob-vote)',
                 linewidth=2)
    best_clip_ep = results.get('best_clip_epoch', results.get('best_epoch'))
    plt.axvline(x=best_clip_ep, color='gray', linestyle='--',
                label=f'Best clip epoch ({best_clip_ep})')
    plt.xlabel('Epoch')
    plt.ylabel('Value')
    plt.title(config['model_name'])
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path / 'curves.png', dpi=200, bbox_inches='tight')
    plt.close()
    print(f'Saved to: {save_path}')

In [18]:
# Step 12: Single-run wrapper
#
# Usage: edit LSTM_CONFIG above, then call run_experiment().
# Everything else cascades from the config dict.

def run_experiment(config, dataframe, audio_dir, test_fold, device=DEVICE):
    """Build loaders -> instantiate model -> train -> save. One call per experiment row."""
    print(f'\n{"="*60}')
    print(f'  {config["model_name"]}  |  fold {test_fold}')
    print(f'  hidden={config["hidden_size"]}  layers={config["num_layers"]}  '
          f'bidir={config["bidirectional"]}  pooling={config["pooling"]}')
    print(f'{"="*60}')

    train_loader, test_loader = get_train_test_loaders(
        dataframe=dataframe,
        audio_dir=audio_dir,
        test_fold=test_fold,
        variant_name=config['variant_name'],
        batch_size=config['batch_size'],
    )
    model = build_lstm_from_config(config, device)
    history, results = run_model_lstm(model, train_loader, test_loader, config, device)
    results['fold'] = test_fold
    save_results(history, results, config)
    return history, results

In [19]:
# Step 13: Baseline validation — Run 1 architecture on short AND long (fold 1)
#
# Purpose: pipeline health check before launching the full experiment grid.
# For each of VARIANT_LIST = ['short', 'long'] the Run 1 baseline architecture
# is trained on fold 1 only. Each run is named `<base>_<variant>` so outputs
# never collide.
#
# Run 1 architecture (rationale in LSTM Experiments.md):
#   hidden=128, layers=2, bidirectional=True, pooling='final'
#
# Expected signals (see LSTM Experiments.md → "What to look for"):
#   * no NaN loss in either variant
#   * short fold-1 clip-level acc roughly in 0.40-0.55
#   * long fold-1 clip-level acc comparable-or-better than short
#
# If both variants pass, proceed to `LSTM experiments.ipynb` for the
# full 9-architecture grid and 5-fold promotion.

BASELINE_ARCH = {
    'base_model_name': 'LSTM_Run1_Baseline',
    'hidden_size':     128,
    'num_layers':      2,
    'bidirectional':   True,
    'pooling':         'final',
}
VARIANT_LIST = ['short', 'long']

baseline_results = []
for variant in VARIANT_LIST:
    run_name = f"{BASELINE_ARCH['base_model_name']}_{variant}"
    config = {
        **LSTM_CONFIG,
        'model_name':    run_name,
        'variant_name':  variant,
        'hidden_size':   BASELINE_ARCH['hidden_size'],
        'num_layers':    BASELINE_ARCH['num_layers'],
        'bidirectional': BASELINE_ARCH['bidirectional'],
        'pooling':       BASELINE_ARCH['pooling'],
    }
    print(f'\n=== Baseline validation: {run_name} (fold 1) ===')
    _, results = run_experiment(config, df, AUDIO_DIR, test_fold=1)
    results['variant']          = variant
    results['base_model_name']  = BASELINE_ARCH['base_model_name']
    baseline_results.append(results)

baseline_summary = pd.DataFrame(baseline_results)[[
    'model', 'variant',
    'hidden_size', 'num_layers', 'bidirectional', 'pooling',
    'best_clip_test_acc', 'best_clip_epoch', 'final_clip_test_acc',
    'best_test_acc', 'best_epoch', 'final_test_acc',
    'training_time_secs',
]]

print('\n--- Baseline validation summary (fold 1, ranked by clip-level acc) ---')
display(baseline_summary.sort_values('best_clip_test_acc', ascending=False))

print('\nReference points for ESC-50 (clip-level, from Piczak 2015):')
print('  Random-forest baseline (B):          0.4400')
print('  CNN SM / SP (short, 41 frames):      ~0.58-0.62')
print('  CNN LM / LP (long, 101 frames):       0.6450')

print('\nHealth criteria:')
print('  * both variants completed without NaN loss')
print('  * clip-level accuracy > 0.44 (beats Piczak baseline B)')
print('  * short fold-1 clip acc roughly in 0.40-0.55 range')
print('If both pass, proceed to `LSTM experiments.ipynb`.')


=== Baseline validation: LSTM_Run1_Baseline_short (fold 1) ===

  LSTM_Run1_Baseline_short  |  fold 1
  hidden=128  layers=2  bidir=True  pooling=final
  Processed 200/1600 clips...
  Processed 400/1600 clips...
  Processed 600/1600 clips...
  Processed 800/1600 clips...
  Processed 1000/1600 clips...
  Processed 1200/1600 clips...
  Processed 1400/1600 clips...
  Processed 1600/1600 clips...
  Total segments: 14400 from 1600 clips
  Processed 200/400 clips...
  Processed 400/400 clips...
  Total segments: 3600 from 400 clips
[LSTM_Run1_Baseline_short] Epoch 001/60 | loss 2.9497 | train 0.2128 | seg 0.2033 | clip 0.2475 | best clip 0.2475 (ep 1)
[LSTM_Run1_Baseline_short] Epoch 005/60 | loss 1.7040 | train 0.5119 | seg 0.3000 | clip 0.4025 | best clip 0.4250 (ep 3)
[LSTM_Run1_Baseline_short] Epoch 010/60 | loss 1.1969 | train 0.6556 | seg 0.3283 | clip 0.4600 | best clip 0.4825 (ep 9)
[LSTM_Run1_Baseline_short] Epoch 015/60 | loss 0.9487 | train 0.7226 | seg 0.3533 | clip 0.4750 | bes

,model,variant,hidden_size,num_layers,bidirectional,pooling,best_clip_test_acc,best_clip_epoch,final_clip_test_acc,best_test_acc,best_epoch,final_test_acc,training_time_secs
0,LSTM_Run1_Baseline_short,short,128,2,True,final,0.5275,46,0.5125,0.384444,46,0.372778,112.6
1,LSTM_Run1_Baseline_long,long,128,2,True,final,0.5125,24,0.4750,0.426250,29,0.410833,311.9



Reference points for ESC-50 (clip-level, from Piczak 2015):
  Random-forest baseline (B):          0.4400
  CNN SM / SP (short, 41 frames):      ~0.58-0.62
  CNN LM / LP (long, 101 frames):       0.6450

Health criteria:
  * both variants completed without NaN loss
  * clip-level accuracy > 0.44 (beats Piczak baseline B)
  * short fold-1 clip acc roughly in 0.40-0.55 range
If both pass, proceed to `LSTM experiments.ipynb`.
